# 세션 브라우저 만들기

에이전트를 제품(데스크톱 앱, IDE 확장, 사내 챗봇)으로 내놓으면 사용자가 가장 먼저 요구하는 것이 사이드바입니다. 지난주 화요일의 대화를 보고, 거기로 되돌아가고, 원본을 잃지 않으면서 새 방향으로 갈라져 나가고 싶어 합니다. 에이전트 루프는 제품의 절반이고, 나머지 절반이 세션 관리입니다.

Claude Agent SDK는 모든 대화를 디스크의 JSONL 기록으로 남깁니다. 그 기록을 다시 읽어 정리해 주는 함수들도 함께 제공하므로, 파일 파서를 짜거나 `~/.claude/projects/`를 손으로 훑지 않고도 그 사이드바를 만들 수 있습니다.

**이 쿡북을 마치면 다음을 할 수 있습니다.**

- 프로젝트의 지난 세션을 브랜치, 제목, 마지막 수정 시각 같은 메타데이터와 함께 페이지 단위로 나열하고 표시하기
- 에이전트를 띄우지 않고 저장된 세션의 메시지를 UI로 다시 읽어 오기
- 세션의 이름을 바꾸고, 태그를 붙이고, 걸러 내어 사용자가 이력을 정리할 수 있게 하기
- 어느 지점에서든 세션을 분기해 그 분기를 살아 있는 `query()` 호출로 재개하기

Claude Code 데스크톱과 VS Code 확장의 세션 사이드바가 바로 이 패턴입니다. 같은 기본 요소로 Agent SDK 위에 어떤 UI든 올릴 수 있습니다.

## 사전 준비

이 가이드를 따라 하기 전에 다음을 확인하세요.

**필요한 사전 지식**

- `async`/`await`을 포함한 Python 기초
- Agent SDK `query()` 함수에 대한 기본적인 친숙함(소개는 [노트북 00](00_The_one_liner_research_agent.ipynb) 참고)

**필요한 도구**

- Python 3.11 이상
- Claude Code CLI (`npm install -g @anthropic-ai/claude-code`)
- Anthropic API 키 ([여기서 발급](https://console.anthropic.com))

## 준비

필요한 의존성을 설치합니다. 세션 관리 함수는 `claude-agent-sdk` v0.1.51에 들어왔습니다.

In [ ]:
%%capture
%pip install -U "claude-agent-sdk>=0.1.51" python-dotenv pandas

`.env`에서 API 키를 불러오고 모델을 설정합니다. 데모 세션이 짧고 저렴하고 빠르기를 원하므로 여기서는 Haiku를 씁니다. 실제 제품에서는 여러분의 에이전트에 맞는 모델을 고르면 됩니다.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-haiku-4-5"

# All demo sessions live under this project directory. Using a dedicated
# cwd keeps the demo isolated from your real Claude Code sessions.
# Note: this path resolves relative to the kernel's working directory
# (claude_agent_sdk/ when launched per the README).
DEMO_DIR = str(Path("session_browser_demo").resolve())
os.makedirs(DEMO_DIR, exist_ok=True)
print(f"Demo project dir: {DEMO_DIR}")

# 1부: 관리할 세션 만들기

세션 관리 함수는 `query()`(또는 Claude Code CLI)가 이미 기록해 둔 대화 기록을 읽습니다. 둘러볼 대상을 만들기 위해 먼저 짧은 대화 세 개를 실행하고 그 세션 ID를 확보하겠습니다.

`cwd` 옵션은 이 대화가 어느 프로젝트 디렉터리에 속하는지 SDK에 알려 줍니다. 대화 기록은 `~/.claude/projects/<encoded-cwd>/<session-id>.jsonl`에 저장되므로, 같은 `cwd`로 호출한 것들은 모두 같은 묶음에 들어갑니다. 나중에 `list_sessions(directory=...)`가 읽는 것이 바로 그 묶음입니다.

토큰 사용을 최소화하기 위해 도구를 비활성화하고 각 실행을 한 턴으로 제한합니다.

In [ ]:
from claude_agent_sdk import ClaudeAgentOptions, ResultMessage, query


async def run_one_turn(prompt: str) -> str:
    """Run a single-turn conversation and return its session_id."""
    opts = ClaudeAgentOptions(
        model=MODEL,
        cwd=DEMO_DIR,
        max_turns=1,
        allowed_tools=[],  # text-only, no tool loop
    )
    session_id = None
    async for msg in query(prompt=prompt, options=opts):
        if isinstance(msg, ResultMessage):
            session_id = msg.session_id
            preview = (msg.result or "")[:80]
            print(f"[{session_id[:8]}] {preview}...")
    if session_id is None:
        raise RuntimeError("No ResultMessage received; check API key and SDK version.")
    return session_id

프롬프트 셋, 세션 셋입니다. 실제 제품이라면 사용자가 에이전트에 물어본 것들이 되겠죠.

In [ ]:
prompts = [
    "Give me three name ideas for a CLI tool that manages git worktrees.",
    "Explain the difference between a mutex and a semaphore in one paragraph.",
    "Write a haiku about merge conflicts.",
]

demo_session_ids = []
for p in prompts:
    sid = await run_one_turn(p)
    demo_session_ids.append(sid)

print(f"\nCreated {len(demo_session_ids)} sessions.")

# 2부: 세션 나열하고 살펴보기

## 세션 목록 만들기

`list_sessions()`는 프로젝트의 대화 기록 디렉터리를 훑어 각 세션의 메타데이터를 최신순으로 반환합니다. 파일 통계와 각 기록의 앞뒤 일부만 읽으므로 디렉터리에 파일이 수백 개 있어도 빠르게 동작합니다. 하위 프로세스를 띄우지 않고 API 호출도 하지 않습니다.

각 `SDKSessionInfo`는 선택 화면에 한 줄을 그리는 데 필요한 것들을 담고 있습니다. 표시용 요약, 마지막 수정 시각, git 브랜치, 작업 디렉터리, 그리고 여러분이 설정한 커스텀 제목이나 태그입니다.

In [ ]:
from datetime import datetime

import pandas as pd

from claude_agent_sdk import SDKSessionInfo, list_sessions

sessions = list_sessions(directory=DEMO_DIR)

# Render as a table. In a real UI this would be your sidebar component.
rows = []
for s in sessions:
    rows.append(
        {
            "id": s.session_id[:8],
            "summary": (s.summary[:50] + "...") if len(s.summary) > 50 else s.summary,
            "modified": datetime.fromtimestamp(s.last_modified / 1000).strftime("%H:%M:%S"),
            "branch": s.git_branch or "-",
            "tag": s.tag or "-",
        }
    )

pd.DataFrame(rows)

이력이 길다면 `limit`과 `offset`을 넘겨 결과를 페이지 단위로 넘길 수 있습니다. 세션 선택 화면은 보통 첫 페이지를 불러온 뒤 사용자가 스크롤하면 더 가져옵니다:

```python
page_2 = list_sessions(directory=DEMO_DIR, limit=20, offset=20)
```

앱이 이미 세션 ID를 저장해 두었고(예를 들어 여러분의 데이터베이스에 사용자 레코드와 함께) 그 한 줄만 필요하다면, 전부 나열하는 것보다 `get_session_info()`가 저렴합니다.

In [ ]:
from claude_agent_sdk import get_session_info

info = get_session_info(demo_session_ids[0], directory=DEMO_DIR)

print(f"Session:      {info.session_id}")
print(f"Summary:      {info.summary}")
print(f"First prompt: {info.first_prompt}")
print(f"Created:      {datetime.fromtimestamp(info.created_at / 1000)}")
print(f"Size:         {info.file_size:,} bytes")

## 세션의 메시지 읽기

사용자가 사이드바에서 세션을 클릭하면 그 대화를 본 화면에 불러옵니다. `get_session_messages()`는 대화 기록에서 메시지 사슬을 재구성해 사용자와 어시스턴트의 턴을 순서대로 반환합니다. 목록 함수와 마찬가지로 순수한 파일 읽기여서 에이전트가 실행 중일 필요가 없습니다.

각 `SessionMessage`에는 `type`(`"user"` 또는 `"assistant"`), `uuid`, 그리고 Anthropic Messages API와 같은 형태(`role`, `content`)의 `message` 딕셔너리가 있습니다.

In [ ]:
from claude_agent_sdk import get_session_messages

messages = get_session_messages(demo_session_ids[0], directory=DEMO_DIR)

for m in messages:
    role = m.type
    # content is a list of blocks; pull out the text ones
    text_parts = [
        b.get("text", "")
        for b in m.message.get("content", [])
        if isinstance(b, dict) and b.get("type") == "text"
    ]
    text = " ".join(text_parts).strip()
    print(f"[{role:>9}] {text[:100]}")

긴 세션에서는 `limit`과 `offset`으로 한 번에 한 구간씩 불러올 수 있습니다. 채팅 화면이라면 열 때 마지막 50개를 불러오고, 사용자가 위로 스크롤하면 이전 페이지를 가져오는 식입니다. 오프셋은 시간순(오래된 것부터)으로 적용되므로 0페이지가 대화의 시작입니다.

# 3부: 제목과 태그로 정리하기

## 세션 이름 바꾸기

자동 생성된 요약은 훑어보기에는 충분하지만, 사용자는 세션에 제대로 된 이름을 붙이고 싶어 하는 경우가 많습니다. `rename_session()`은 대화 기록에 제목 항목을 덧붙이고, 다음 읽기에서 `list_sessions()`가 이를 `custom_title`로 가져옵니다.

덧붙이기는 저렴하고 멱등적입니다. 이름 바꾸기를 두 번 호출하면 나중 제목이 이깁니다. 파일을 다시 쓰는 일은 일어나지 않습니다.

In [ ]:
from claude_agent_sdk import rename_session

rename_session(demo_session_ids[0], "Worktree CLI naming brainstorm", directory=DEMO_DIR)
rename_session(demo_session_ids[2], "Haiku corner", directory=DEMO_DIR)

# Verify the titles stuck
for s in list_sessions(directory=DEMO_DIR):
    label = s.custom_title or "(auto)"
    print(f"{s.session_id[:8]}  custom_title={label!r}  summary={s.summary[:40]!r}")

## 태그 붙이고 걸러 내기

태그는 세션에 붙는 문자열 하나입니다. 제품에 필요한 어떤 분류에든 쓰세요. `"archived"`, `"needs-review"`, `"favorite"` 같은 식입니다. 태그를 지우려면 `None`을 넘기세요.

흔한 패턴은 소프트 삭제입니다. 대화 기록 파일을 지우는 대신 `"__hidden"` 태그를 붙이고 목록 화면에서 걸러 내는 것이죠. 데이터는 복구 가능한 상태로 남습니다.

In [ ]:
from claude_agent_sdk import tag_session

# Mark two sessions as favorites, hide the other
tag_session(demo_session_ids[0], "favorite", directory=DEMO_DIR)
tag_session(demo_session_ids[2], "favorite", directory=DEMO_DIR)
tag_session(demo_session_ids[1], "__hidden", directory=DEMO_DIR)


def visible_sessions(directory: str, tag_filter: str | None = None) -> list[SDKSessionInfo]:
    """List sessions, hiding soft-deletes and optionally filtering by tag."""
    results = []
    for s in list_sessions(directory=directory):
        if s.tag == "__hidden":
            continue
        if tag_filter is not None and s.tag != tag_filter:
            continue
        results.append(s)
    return results


favorites = visible_sessions(DEMO_DIR, tag_filter="favorite")
print(f"Visible favorites: {len(favorites)}")
for s in favorites:
    print(f"  {s.session_id[:8]}  [{s.tag}]  {s.custom_title or s.summary}")

태그는 리스트가 아니라 단일 값입니다. 여러 축이 필요하다면(예를 들어 상태와 범주), `"review:urgent"`처럼 하나의 문자열에 인코딩해 읽을 때 파싱하거나, `session_id`를 키로 삼아 여러분의 데이터베이스에 더 풍부한 상태를 저장하세요.

# 4부: 분기와 재개

## 기존 대화에서 갈라져 나가기

분기(fork)는 세션의 대화 기록을 새 파일로 복사하면서 메시지 ID를 새로 매깁니다. 원본은 그대로 남습니다. "다른 방식으로 시도해 보기" 기능의 바탕이 되는 기본 요소로, 사용자는 원래 스레드를 간직한 채 실험할 새 스레드를 얻습니다.

`fork_session()`은 새 파일을 쓰고 그 ID를 반환합니다. 에이전트를 실행하지는 않으므로, `query()`로 재개하기 전까지 분기본은 디스크에 놓여 있습니다.

In [ ]:
from claude_agent_sdk import fork_session

source = demo_session_ids[0]

fork = fork_session(
    source,
    directory=DEMO_DIR,
    title="Worktree CLI names (round 2)",
)

print(f"Source: {source}")
print(f"Fork:   {fork.session_id}")

# The fork starts with the same message history as the source
source_msgs = get_session_messages(source, directory=DEMO_DIR)
fork_msgs = get_session_messages(fork.session_id, directory=DEMO_DIR)
print(f"Source has {len(source_msgs)} messages, fork has {len(fork_msgs)}")

전체 이력이 아니라 특정 지점에서 갈라져 나가려면 `up_to_message_id`를 넘기세요. 분기본에는 그 메시지까지 포함한 원본 기록이 담깁니다. 메시지 UUID는 `get_session_messages()[i].uuid`에서 얻을 수 있습니다.

## 분기본을 살아 있는 질의로 재개하기

분기본은 그저 대화 기록 파일일 뿐입니다. 다시 실행 중인 대화로 만들려면 그 ID를 `ClaudeAgentOptions.resume`에 넘기세요. 에이전트가 분기된 이력을 불러와 거기서부터 이어 갑니다.

In [ ]:
resume_opts = ClaudeAgentOptions(
    model=MODEL,
    cwd=DEMO_DIR,
    max_turns=1,
    allowed_tools=[],
    resume=fork.session_id,
)

async for msg in query(
    prompt="Those were okay. Give me three more names, but punnier.",
    options=resume_opts,
):
    if isinstance(msg, ResultMessage):
        print(f"[fork {fork.session_id[:8]} resumed]")
        print(msg.result)

원본 세션은 그대로 남아 있습니다. 다시 나열해 보면 원본과 분기본이 각자의 이력을 가진 별개의 줄로 보입니다.

In [ ]:
for s in list_sessions(directory=DEMO_DIR):
    marker = "(fork)" if s.session_id == fork.session_id else "      "
    print(f"{marker} {s.session_id[:8]}  {s.custom_title or s.summary[:50]}")

# 정리

`delete_session()`은 대화 기록 파일을 제거합니다. 되돌릴 수 없는 삭제이므로, 사용자에게 보이는 UI에서는 보통 3부의 소프트 삭제 태그 패턴이 더 안전한 기본값입니다.

여기서는 데모가 만든 것들을 정리하는 데 사용합니다.

In [ ]:
from claude_agent_sdk import delete_session

# Clean up every session in the demo dir, including the fork
for s in list_sessions(directory=DEMO_DIR):
    delete_session(s.session_id, directory=DEMO_DIR)
    print(f"Deleted {s.session_id[:8]}")

remaining = list_sessions(directory=DEMO_DIR)
print(f"\n{len(remaining)} session(s) remaining.")

# 정리하며

Agent SDK의 로컬 대화 기록 저장소를 대상으로 세션 브라우저의 핵심을 만들었습니다.

- **나열**: `list_sessions()`가 사이드바를 그리는 데 필요한 모든 것을 제공하며, 전체 기록을 파싱하는 대신 파일 통계와 앞뒤 일부만 읽으므로 규모가 커져도 견딥니다.
- **읽기**: `get_session_messages()`가 에이전트를 띄우지 않고 대화를 다시 불러옵니다.
- **정리**: `rename_session()`과 `tag_session()`이 메타데이터 항목을 덧붙이므로 저렴하고, 가장 최근 호출이 이깁니다.
- **분기**: `fork_session()`과 `options.resume`을 함께 쓰면 사용자가 원본을 건드리지 않고 대화를 갈라 이어 갈 수 있습니다.

이 모두가 `~/.claude/projects/`에 대한 순수한 파일 작업입니다. 에이전트 하위 프로세스가 실행 중이든 아니든 동작하며, Claude Code CLI가 기록하는 것과 같은 대화 기록을 봅니다.

## 다음으로 갈 곳

- **UI에 연결하기.** 이 함수들은 UI에 구애받지 않습니다. FastAPI 라우트나 Electron IPC 핸들러 뒤에 두면 세션 사이드바의 백엔드가 됩니다.
- **여러 호스트에 걸친 세션.** 대화 기록은 로컬 디스크에 있습니다. 여러 머신에서 세션을 공유하려면 파일을 동기화하거나 세션 ID와 메시지를 여러분의 저장소에 색인하세요. 패턴은 [디스크의 세션 관리](https://docs.claude.com/en/agent-sdk/local-session-management)를 참고하세요.
- **TypeScript.** 같은 API가 `@anthropic-ai/claude-agent-sdk`에 카멜 표기법(`listSessions`, `forkSession` 등)으로 있습니다. [TypeScript SDK 레퍼런스](https://docs.claude.com/en/agent-sdk/typescript)를 참고하세요.
- **큰 그림.** 노트북 [00](00_The_one_liner_research_agent.ipynb)부터 [03](03_The_site_reliability_agent.ipynb)까지가 에이전트 자체를 만드는 법을 다룹니다. 이 노트북은 그것이 남긴 것을 관리하는 법을 다룹니다.